# Assignment 1: Word Embeddings, Sentiments and Topics

**Course:** Large Language Models for Marketing (FEM11154)  
**Dataset:** Yelp Academic Dataset (15,000 reviews)  
**DV:** Star rating (1–5)

This notebook runs the full analysis pipeline and serves as the submission appendix.

---
**Sections:**
1. Setup & Data Loading
2. Word Embeddings (Part 2)
3. Topic Modelling & Sentiment Analysis (Part 4)
4. Drivers of Star Ratings — Regression (Part 5)

In [ ]:
import os, sys
# Add scripts folder to path so we can import the modules
sys.path.insert(0, os.path.join('..', 'scripts'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = os.path.join('..', 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Environment ready.')

---
## Part 1 · Dataset

**Source:** Yelp Academic Dataset — `yelp_review.csv` (Kaggle) or `yelp_academic_dataset_review.json` (official).  
We sample **15,000 reviews** stratified across star ratings (3,000 per star level).

**Criteria check:**
- ≥ 5,000 observations ✓ (15,000)
- Multi-sentence documents ✓ (Yelp reviews average 3–5 sentences)
- Numeric DV ✓ (star rating 1–5)

In [ ]:
import sys, importlib
sys.path.insert(0, os.path.join('..', 'scripts'))
import importlib, data_prep_01 as dp

processed_path = os.path.join('..', 'data', 'yelp_processed.csv')

if not os.path.exists(processed_path):
    print('Downloading dataset and running data prep...')
    dp.main()
else:
    print(f'Processed data already exists at {processed_path}')

df = pd.read_csv(processed_path)
print(f'Loaded {len(df):,} rows')
df.head(3)

In [ ]:
# Dataset statistics
print('Star rating distribution:')
print(df['stars'].value_counts().sort_index())
print(f'\nAverage document length (words): {df["tokens"].str.split().str.len().mean():.1f}')
print(f'Median document length (words): {df["tokens"].str.split().str.len().median():.0f}')

---
## Part 2 · Word Embeddings

We train a **Word2Vec Skip-gram** model (`vector_size=50`, `window=5`, `min_count=5`) on the tokenised corpus.

### Why Word2Vec over GloVe?
- Word2Vec trains directly on our corpus, capturing domain-specific co-occurrences in Yelp reviews.
- GloVe pre-trained models are trained on general corpora (Wikipedia, Common Crawl) and may miss Yelp-specific vocabulary (e.g., "foie gras", "valet", "prix fixe").

In [ ]:
import embeddings_02 as emb

model_path = os.path.join(OUTPUT_DIR, 'word2vec_model.bin')

if not os.path.exists(model_path):
    print('Training Word2Vec...')
    w2v_model = emb.train_word2vec(df)
    w2v_model.save(model_path)
else:
    from gensim.models import Word2Vec
    print('Loading saved Word2Vec model...')
    w2v_model = Word2Vec.load(model_path)

emb.inspect_model(w2v_model)

### 2.1 Sentiment Direction

We define a **sentiment direction** as `vec('good') − vec('bad')` and project the entire vocabulary onto this axis.  
Words at the positive end of this axis semantically cluster around praise and satisfaction; words at the negative end around complaints and disappointment.

In [ ]:
direction = emb.sentiment_direction_analysis(w2v_model, top_n=15)

img = mpimg.imread(os.path.join(OUTPUT_DIR, 'fig_sentiment_axis.png'))
plt.figure(figsize=(7, 9))
plt.imshow(img); plt.axis('off')
plt.title('Sentiment Axis — Top/Bottom Vocabulary Projections')
plt.tight_layout(); plt.show()

### 2.2 Dimension Interpretation

We select **Dimension 0** and examine the words with the highest and lowest values on this axis to infer its latent meaning.

In [ ]:
emb.dimension_analysis(w2v_model, dim=0, top_n=15)

### 2.3 Interesting Analysis 1: Word Analogies

Using the **3CosAdd** method (king − man + woman ≈ queen), we test whether the embedding space encodes meaningful semantic relationships relevant to the restaurant/hospitality domain.

In [ ]:
emb.word_analogies(w2v_model)

### 2.4 Interesting Analysis 2: t-SNE Cluster Visualisation

We select five marketing-relevant seed words and visualise their nearest neighbours in 2-D using **t-SNE**. Clusters that emerge reflect the semantic neighbourhood of each concept in the Yelp review space.

In [ ]:
seed_words = ['service', 'food', 'price', 'ambiance', 'staff']
emb.tsne_cluster_plot(w2v_model, seed_words, n_neighbors=10)

img = mpimg.imread(os.path.join(OUTPUT_DIR, 'fig_tsne_clusters.png'))
plt.figure(figsize=(11, 8))
plt.imshow(img); plt.axis('off')
plt.tight_layout(); plt.show()

---
## Part 4 · Topic Modelling & Sentiment Analysis

### Design Choice: Document-level vs Sentence-level Topics

Yelp reviews tend to describe a **single overall dining/service experience** — the reviewer either liked or disliked the restaurant.  
While reviews may mention multiple aspects (food, service, ambiance), these are facets of one coherent experience rather than truly separate topics.  
Therefore we assign **one dominant topic per document** using BERTopic.

We use `all-MiniLM-L6-v2` as the sentence-transformer backbone — it offers a good balance of quality and compute speed.

In [ ]:
import topic_sentiment_03 as ts

topics_path = os.path.join('..', 'data', 'yelp_topics.csv')

if not os.path.exists(topics_path):
    print('Running BERTopic + VADER pipeline...')
    ts.main()
else:
    print(f'Topic data already exists at {topics_path}')

df_topics = pd.read_csv(topics_path)
print(f'Loaded {len(df_topics):,} rows with topic assignments')
df_topics[['text_clean', 'stars', 'topic', 'topic_prob', 'sentiment']].head(5)

### 4.1 Topic Overview

After fitting BERTopic, we inspect the top words per topic and assign human-readable marketing labels.  
See `outputs/topic_labels.txt` for the full list of topics and their representative words.

In [ ]:
# Display topic distribution
topic_counts = df_topics[df_topics['topic'] != -1]['topic'].value_counts().head(15)
print('Top 15 topics by document count:')
print(topic_counts)

# Display topic prevalence figure
img = mpimg.imread(os.path.join(OUTPUT_DIR, 'fig_topic_prevalence.png'))
plt.figure(figsize=(10, 6))
plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()

### 4.2 Topic-Level Sentiment

We compute VADER compound scores (−1 = very negative, +1 = very positive) for each document and aggregate to the topic level.

In [ ]:
agg = ts.topic_level_sentiment(df_topics)
print('Topic-level sentiment (mean VADER compound score):')
print(agg.to_string(index=False))

img = mpimg.imread(os.path.join(OUTPUT_DIR, 'fig_topic_sentiment.png'))
plt.figure(figsize=(10, 6))
plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()

### 4.3 Subgroup Comparison: High (4–5★) vs Low (1–2★) Ratings

We compare **topic prevalence** and **topic-level sentiment** across high and low rating subgroups to identify which aspects of the dining experience drive positive vs negative reviews.

In [ ]:
img = mpimg.imread(os.path.join(OUTPUT_DIR, 'fig_subgroup_comparison.png'))
plt.figure(figsize=(15, 6))
plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()

**Interpretation template** (fill in after inspecting your topic labels):

> - Topics associated with **food quality** are more prevalent in high-rating reviews and carry strongly positive sentiment.
> - Topics related to **wait times / service failures** are more prevalent in low-rating reviews.
> - The sentiment gap between high and low raters is widest for topics about **[your observation]**, suggesting that **[managerial insight]**.

---
## Part 5 · Drivers of the Dependent Variable

We estimate three OLS models:

| Model | Predictors | Purpose |
|-------|-----------|--------|
| A | `sentiment_overall` | Baseline: does sentiment predict star ratings? |
| B | `topic_1 … topic_K` | Which topics directly drive star ratings? |
| C | `sentiment + topics + sentiment×topic` | Which topics amplify or dampen the sentiment effect? |

**Variable definitions:**
- `sentiment_centered`: VADER compound score, mean-centred (μ=0)
- `topic_k`: standardised indicator (1 if doc assigned to topic k), σ=1
- `sent_x_topic_k`: product of mean-centred sentiment and standardised topic indicator

In [ ]:
import regression_04 as reg

df_reg, top_topics = reg.build_features(df_topics)
print(f'Regression dataset: {len(df_reg):,} docs, {len(top_topics)} topic features')

In [ ]:
y          = df_reg['stars']
sent       = df_reg[['sentiment_centered']]
topic_cols = [f'topic_{k}' for k in top_topics]
inter_cols = [f'sent_x_topic_{k}' for k in top_topics]

import statsmodels.api as sm

model_a = reg.run_ols(y, sent, 'Model A: stars ~ sentiment_overall')

In [ ]:
model_b = reg.run_ols(y, df_reg[topic_cols], 'Model B: stars ~ topic_1 ... topic_K')

In [ ]:
X_c     = pd.concat([sent, df_reg[topic_cols], df_reg[inter_cols]], axis=1)
model_c = reg.run_ols(y, X_c, 'Model C: stars ~ sentiment + topics + interactions')

In [ ]:
reg.plot_coefficients(model_c, 'Model C — OLS Coefficients (95% CI)')

img = mpimg.imread(os.path.join(OUTPUT_DIR, 'fig_model_c_coefs.png'))
plt.figure(figsize=(9, 7))
plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()

In [ ]:
reg.print_managerial_insights(model_a, model_b, model_c, top_topics)

---
## Summary of Managerial Implications

> *Fill in this section based on your actual results after running the notebook.*

**1. Sentiment drives ratings overall (Model A)**  
A higher VADER compound score is associated with higher star ratings (β > 0, p < 0.05). This confirms that reviewer language closely mirrors their satisfaction level.

**2. Some topics matter more than others (Model B)**  
Topics related to [Topic X — e.g., food quality] have the strongest positive association with star ratings. Topics about [Topic Y — e.g., wait times] carry the strongest negative association.

**3. Sentiment is not equally impactful across topics (Model C)**  
The interaction analysis reveals that sentiment has a disproportionately large effect on ratings *when the review focuses on [topic]*. Managers should pay close attention to emotional language in reviews about these areas.

**Actionable recommendation:**  
Monitoring sentiment in reviews tagged to high-impact topics (identified in Model C) provides an early-warning signal for rating deterioration — more valuable than monitoring overall sentiment alone.